### Vamos a probar la nueva data en español

In [1]:
import pandas as pd
import sys
import os
import re
sys.path.append(os.path.abspath('../scripts'))  #esto lo colocamos para que el python nos lea la ruta bien
from utils import process_null_values, tokenice_and_lemati # Importa la funcinn desde utils
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [2]:
MultiLeng = pd.read_csv('../data/spanish/train.csv')
MultiLeng.describe()

,Unnamed: 0,stars
count,1.200000e+06,1.200000e+06
mean,5.999995e+05,3.000000e+00
std,3.464103e+05,1.414214e+00
min,0.000000e+00,1.000000e+00
25%,2.999998e+05,2.000000e+00
50%,5.999995e+05,3.000000e+00
75%,8.999992e+05,4.000000e+00
max,1.199999e+06,5.000000e+00


In [3]:
MultiLeng.head()

,Unnamed: 0,review_id,product_id,reviewer_id,stars,review_body,review_title,language,product_category
0,0,de_0203609,product_de_0865382,reviewer_de_0267719,1,Armband ist leider nach 1 Jahr kaputt gegangen,Leider nach 1 Jahr kaputt,de,sports
1,1,de_0559494,product_de_0678997,reviewer_de_0783625,1,In der Lieferung war nur Ein Akku!,EINS statt ZWEI Akkus!!!,de,home_improvement
2,2,de_0238777,product_de_0372235,reviewer_de_0911426,1,"Ein Stern, weil gar keine geht nicht. Es hande...",Achtung Abzocke,de,drugstore
3,3,de_0477884,product_de_0719501,reviewer_de_0836478,1,"Dachte, das wären einfach etwas festere Binden...",Zu viel des Guten,de,drugstore
4,4,de_0270868,product_de_0022613,reviewer_de_0736276,1,Meine Kinder haben kaum damit gespielt und nac...,Qualität sehr schlecht,de,toy


### Eliminando fiilas innecesarias

In [4]:
MultiLeng = MultiLeng.drop(MultiLeng[(MultiLeng['language'] != 'en') & (MultiLeng['language'] != 'es')].index)

In [5]:
conteo_reseñas = MultiLeng.groupby('language').size()
conteo_reseñas

language
en    200000
es    200000
dtype: int64

## Limpiando datos

In [6]:
process_null_values(MultiLeng)

Total de registros:
la columna review_title tiene un Total de: 33 valores vacios
--------------------------------------------------------------------------------
Porcentaje de registros:
la columna review_title tiene un Porcentaje de: 0.00825 valores vacios
--------------------------------------------------------------------------------
Si las columnas son irrelevantes, podemos hacer algo con sus datos, que te gustaria eliminarlos o los conservamos?
columna review_title eliminada exitosamente
data limpiada maestro, que tengas un buen dia :)


## Eliminando columnas innecesarias

In [7]:
MultiLeng.drop(['review_id'], axis=1, inplace=True)
MultiLeng.drop(['product_id'], axis=1, inplace=True)
MultiLeng.drop(['reviewer_id'], axis=1, inplace=True)
MultiLeng.drop(['product_category'], axis=1, inplace=True)
MultiLeng.drop(['Unnamed: 0'], axis=1, inplace=True)


In [8]:
MultiLeng.head()

,stars,review_body,language
200000,1,Arrived broken. Manufacturer defect. Two of th...,en
200001,1,the cabinet dot were all detached from backing...,en
200002,1,I received my first order of this product and ...,en
200003,1,This product is a piece of shit. Do not buy. D...,en
200004,1,went through 3 in one day doesn't fit correct ...,en


### divideremos la data en espaniol e ingles, por si acaso la de ingles no es de ayuda

In [9]:
DataSpanish = MultiLeng[(MultiLeng['language'] == 'es')]
DataEnglish = MultiLeng[(MultiLeng['language'] == 'en')]


In [10]:
DataSpanish.head()


,stars,review_body,language
400000,1,Nada bueno se me fue ka pantalla en menos de 8...,es
400001,1,"Horrible, nos tuvimos que comprar otro porque ...",es
400002,1,Te obligan a comprar dos unidades y te llega s...,es
400003,1,"No entro en descalificar al vendedor, solo pue...",es
400004,1,Llega tarde y co la talla equivocada,es


## VISUALIZAR DANTIDAD DE DE LOS PUNTAJES

In [15]:
#antes de pasar a hacer la lematizacion hare una eliminacion de muchos datos positivos que hay(no hay de otra)
#eliminar muchos registros para balancear la data
condition = DataSpanish['stars'] == 1
num_to_remove = 8000
rows_to_remove = DataSpanish[condition].head(num_to_remove)

# Eliminar esas filas del DataFrame original
DataSpanish = DataSpanish.drop(rows_to_remove.index)

DataSpanish.describe()


,stars
count,156000.000000
mean,2.987179
std,1.418686
min,1.000000
25%,2.000000
50%,3.000000
75%,4.000000
max,5.000000


In [16]:
conteo_reseñas = DataSpanish.groupby('stars').size()
conteo_reseñas

stars
1    32000
2    31000
3    31000
4    31000
5    31000
dtype: int64

In [17]:
DataSpanish['processed_text'] = DataSpanish['review_body'].apply(lambda x: re.sub(r'[^\w\sáéíóúñÁÉÍÓÚÑ]', '', x))

In [18]:
DataSpanish.head()

,stars,review_body,language,processed_text
408000,1,No lee nada. Ni usb ni mp3 ni nada. Un engaño ...,es,No lee nada Ni usb ni mp3 ni nada Un engaño en...
408001,1,En mi opinión es mejor el manual q venden en m...,es,En mi opinión es mejor el manual q venden en m...
408002,1,"Era para regalar a mi hermano, dice que muy du...",es,Era para regalar a mi hermano dice que muy dur...
408003,1,Me costó un poko instalar no lo puedo utilizar,es,Me costó un poko instalar no lo puedo utilizar
408004,1,"Muy mala tela. Se queda toda la pelusa pegada,...",es,Muy mala tela Se queda toda la pelusa pegada y...


In [19]:
# Procesar la columna 'review_body' en lote
DataSpanish['processed_text'] = tokenice_and_lemati(DataSpanish['processed_text'], process=2, language='es')

In [22]:
analyzer = SentimentIntensityAnalyzer()

def analizar_vader(texto):
    # Obtener el puntaje de polaridad
    sentiment_score = analyzer.polarity_scores(texto)
    compound_score = sentiment_score['compound']

    # Evaluar el sentimiento en base al puntaje "compound"
    if compound_score >= 0.5:
        return 'contento'  # Muy positivo
    elif compound_score >= 0.05:
        return 'positivo'  # Positivo
    elif compound_score <= -0.8:
        return 'enojado'  # Muy negativo
    elif compound_score <= -0.5:
        return 'insatisfecho'  #"enojado"
    elif compound_score <= -0.05:
        return 'negativo'  # Negativo
    else:
        return 'neutral'  # Neutral

In [20]:
DataSpanish.head()

,stars,review_body,language,processed_text
408000,1,No lee nada. Ni usb ni mp3 ni nada. Un engaño ...,es,No lee Ni usb Un engaño tods regla No dejar qu...
408001,1,En mi opinión es mejor el manual q venden en m...,es,En opinión mejor manual q venden mercadona
408002,1,"Era para regalar a mi hermano, dice que muy du...",es,Era regalar hermano duras difícil moldear Me t...
408003,1,Me costó un poko instalar no lo puedo utilizar,es,Me costó poko instalar no utilizar
408004,1,"Muy mala tela. Se queda toda la pelusa pegada,...",es,Muy mala tela Se queda pelusa pegada lava suel...


In [21]:
DataSpanish.to_csv('../data_process/DataSp1r.csv', index=False)

In [ ]:
DataSpanish['sentiment'] = DataSpanish['processed_text'].apply(analizar_vader)
